# 02 · Inference — RGB → thermal on your own images

Run a trained checkpoint over arbitrary imagery: your own UAV frames, or a
held-out slice of the datasets.

**Point `CHECKPOINT` at a real file first** — either a Notebook Output from
training, or an uploaded Kaggle Dataset containing `best.pt`.

In [ ]:
# --- Pull the project code from GitHub -------------------------------------
# Requires "Internet" to be ON in the notebook settings panel on the right.
REPO_URL = "https://github.com/Astrq23/rgb-2-thermal.git"
BRANCH   = "main"

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/rgb-2-thermal")

if REPO_DIR.exists():
    # Re-running the notebook: fast-forward instead of re-cloning.
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

print("repo:", REPO_DIR)
print("commit:", subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip())

In [ ]:
# Editable install so `import rgb2thermal` works everywhere, including inside
# DataLoader worker processes. --no-deps keeps Kaggle's preinstalled torch.
!pip install -e . --no-deps -q

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
from pathlib import Path

# EDIT THESE TWO
CHECKPOINT = "/kaggle/input/<your-notebook-output>/outputs/pix2pix_uav/checkpoints/best.pt"
INPUT_DIR  = "/kaggle/input/<your-images>"

assert Path(CHECKPOINT).exists(), (
    f"No checkpoint at {CHECKPOINT}.\n"
    "Add data -> Notebook Output -> pick your training run, then fix the path."
)
print("checkpoint OK:", CHECKPOINT)

## Generate

In [ ]:
!python scripts/predict.py \
    --checkpoint {CHECKPOINT} \
    --input {INPUT_DIR} \
    --out /kaggle/working/predictions \
    --colormap inferno \
    --side-by-side

In [ ]:
from pathlib import Path
from IPython.display import Image as ShowImage, display

for path in sorted(Path("/kaggle/working/predictions").glob("*.png"))[:8]:
    print(path.name)
    display(ShowImage(filename=str(path), width=900))

## Options

| Flag | Effect |
|---|---|
| *(none)* | Single-channel grayscale — use this if the output feeds another model |
| `--colormap inferno` | Colourised for human viewing |
| `--side-by-side` | Input next to prediction |
| `--keep-size` | Upscale the 256×256 output back to the input resolution |
| `--domain dronevehicle` | Pick which sensor style to imitate (only if trained with `model.use_domain_embedding: true`) |

## What this model does and does not give you

It predicts **thermal appearance** — the spatial pattern of relative heat.

It does **not** predict temperature. The training pipeline applies a per-image
percentile stretch to every thermal frame, which is exactly what makes three
different thermal cameras trainable as one dataset, and which necessarily
discards absolute radiometric values. A pixel of 200 means "hot relative to this
frame", never "44 °C".

For UAV work that is usually the right trade: augmenting detection or
segmentation training sets, previewing what a thermal payload would see,
prototyping a pipeline before the hardware arrives. It is not a substitute for a
radiometric sensor, and nothing downstream should treat it as one.